# Thalika Studio — Kaggle GPU

**မစခင် (တစ်ကြိမ်သာ):**
1. **Settings → Accelerator → `GPU T4 x2`** ရွေးပါ (P100 မရပါ)။
2. **Settings → Internet → On** (ဖုန်းနံပါတ် verify လုပ်ထားသော account လိုသည်)။
3. **Add-ons → Secrets** တွင် `THALIKA_OWNER_PASSWORD` (စာလုံး ၁၂ လုံးအထက်) ထည့်ပြီး ဒီ notebook ကို attach လုပ်ပါ။
4. Notebook ကို **Private** ထားပါ။ Voice reference နှင့် WAV များ output ထဲသိမ်းသည်။

**Run နည်း:** iPhone ကနေသုံးမယ်ဆိုရင် **Save Version → Save & Run All (Commit)** ကိုနှိပ်ပါ။ Browser ပိတ်ထားလည်း ~11 နာရီ run နေမည်။ Version log ထဲက `https://….trycloudflare.com` link ကို Safari မှာဖွင့်ပါ။

**ယခင် data ပြန်ယူရန်:** ယခင် version ၏ output ကို **Add Input → Your Work → ဒီ notebook** ဖြင့် attach လုပ်ပါ။ အလိုအလျောက် restore လုပ်ပါမည်။

Optional secrets: `CLOUDFLARE_TUNNEL_TOKEN` + `THALIKA_APP_ORIGIN` (fixed URL), `THALIKA_NOTIFY_URL` (link ပို့ရန် https webhook ဥပမာ ntfy.sh topic), `GEMINI_API_KEY`, `HF_TOKEN`, `GITHUB_TOKEN` (private repo ဆိုလျှင်)။


In [ ]:
# Change these only if you use a fork or a different branch.
REPO_URL = "https://github.com/phyoethantnaing/thalika-iphone-cloud.git"
BRANCH = "cloud-iphone"
MAX_HOURS = "11"  # Kaggle stops GPU sessions at 12 h; Thalika stops safely before that.

import os, subprocess, sys
from pathlib import Path

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or ""
    except Exception:
        return ""

repo = Path("/kaggle/tmp/thalika-iphone-cloud") if Path("/kaggle/tmp").is_dir() else Path("/tmp/thalika-iphone-cloud")
if not (repo / ".git").is_dir():
    url = REPO_URL
    token = secret("GITHUB_TOKEN")
    if token:  # private repository: token is used for this clone only and not stored in the remote
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=False)

os.environ["THALIKA_MAX_HOURS"] = MAX_HOURS
subprocess.run([sys.executable, str(repo / "scripts" / "kaggle-start.py")], check=True)
